# Task 4 – Data Science Tool Mastery Project
**Prepared by: Ganesh**

## Project: Wine Classification and Exploratory Analytics

This practical project demonstrates a Python data-science workflow using the Wine recognition dataset distributed with scikit-learn (originally from the UCI Machine Learning Repository). The dataset has 178 observations, 13 numeric chemical measurements, and three wine classes.

**Tools used:** Python, Jupyter Notebook, Pandas, NumPy, Matplotlib, Seaborn, scikit-learn, and SQLite through Python's built-in `sqlite3` module.

**Learning outcomes:** data loading and inspection, data quality checks, descriptive statistics, visualization, SQL queries, preprocessing, model training, and model evaluation.

> This is a classification exercise on a small benchmark dataset. Its model scores should not be interpreted as proof of performance on commercial wine or production data.


## 1. Environment and imports
Run the notebook from top to bottom. The required packages are listed in `requirements.txt`.

In [ ]:
import sys, sqlite3, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)


## 2. Load and understand the dataset

In [ ]:
wine = load_wine(as_frame=True)
df = wine.frame.copy()
df["target_name"] = df["target"].map(dict(enumerate(wine.target_names)))

print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe().T)


## 3. Data quality audit

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique(),
})
print("Exact duplicate rows:", df.duplicated().sum())
display(quality)


The bundled benchmark dataset is already structured and contains no missing values or exact duplicate rows. We still perform checks rather than assuming the data is clean.

## 4. Pandas and NumPy practice

In [ ]:
# Pandas: filtering, sorting, grouping, aggregation
high_alcohol = df.loc[df["alcohol"] > df["alcohol"].mean(),
                      ["alcohol", "malic_acid", "target_name"]]
display(high_alcohol.sort_values("alcohol", ascending=False).head(10))

group_summary = df.groupby("target_name").agg(
    sample_count=("target", "size"),
    mean_alcohol=("alcohol", "mean"),
    mean_color_intensity=("color_intensity", "mean"),
    mean_flavanoids=("flavanoids", "mean")
).round(2)
display(group_summary)

# NumPy: vectorized calculations
alcohol_values = df["alcohol"].to_numpy()
print("Alcohol mean:", round(np.mean(alcohol_values), 3))
print("Alcohol standard deviation:", round(np.std(alcohol_values), 3))
print("Alcohol 25th/50th/75th percentiles:",
      np.percentile(alcohol_values, [25, 50, 75]).round(3))


## 5. Visualization

In [ ]:
sns.set_theme()
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="target_name")
plt.title("Number of observations by wine class")
plt.xlabel("Wine class"); plt.ylabel("Count"); plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 4))
sns.histplot(data=df, x="alcohol", hue="target_name", kde=True, element="step")
plt.title("Alcohol distribution by class"); plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="target_name", y="color_intensity")
plt.title("Color intensity by wine class"); plt.tight_layout(); plt.show()

plt.figure(figsize=(10, 8))
corr = df[wine.feature_names].corr()
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation matrix of numeric features"); plt.tight_layout(); plt.show()


## 6. SQL practice using SQLite (no separate SQL installation required)
Python's built-in `sqlite3` module provides a lightweight local SQL database. This demonstrates SQL querying without MySQL or Power BI.

In [ ]:
conn = sqlite3.connect(":memory:")
df.to_sql("wine", conn, index=False, if_exists="replace")

sql_queries = {
    "row_count": "SELECT COUNT(*) AS total_rows FROM wine",
    "class_counts": """SELECT target_name, COUNT(*) AS samples
                       FROM wine GROUP BY target_name ORDER BY samples DESC""",
    "feature_averages": """SELECT target_name,
                         ROUND(AVG(alcohol), 2) AS avg_alcohol,
                         ROUND(AVG(color_intensity), 2) AS avg_color_intensity
                         FROM wine GROUP BY target_name ORDER BY target_name""",
    "top_alcohol": """SELECT target_name, alcohol FROM wine
                     WHERE alcohol > (SELECT AVG(alcohol) FROM wine)
                     ORDER BY alcohol DESC LIMIT 10"""
}
for name, query in sql_queries.items():
    print("\\n", name.upper())
    display(pd.read_sql_query(query, conn))


## 7. Machine learning: classification
We use a stratified train/test split and a preprocessing pipeline. Standardization is fitted only on training folds through the pipeline, reducing data leakage.

In [ ]:
X = df[wine.feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
])
forest_model = RandomForestClassifier(
    n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced"
)

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": forest_model
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision_macro": precision_score(y_test, pred, average="macro"),
        "recall_macro": recall_score(y_test, pred, average="macro"),
        "f1_macro": f1_score(y_test, pred, average="macro")
    })
results_df = pd.DataFrame(results).set_index("model").round(4)
display(results_df)


## 8. Evaluate the selected model

In [ ]:
best_name = results_df["f1_macro"].idxmax()
best_model = models[best_name]
best_pred = best_model.predict(X_test)

print("Selected by test macro-F1:", best_name)
print(classification_report(y_test, best_pred, target_names=wine.target_names))

ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred, display_labels=wine.target_names, cmap="Blues"
)
plt.title(f"Confusion matrix – {best_name}")
plt.tight_layout(); plt.show()


## 9. Cross-validation check

In [ ]:
# Cross-validation is performed on the training partition only.
cv_scores = cross_val_score(
    logistic_model, X_train, y_train, cv=5, scoring="f1_macro"
)
print("5-fold training CV macro-F1 scores:", np.round(cv_scores, 4))
print("Mean CV macro-F1:", round(cv_scores.mean(), 4))
print("Std CV macro-F1:", round(cv_scores.std(), 4))


## 10. Interpretation, limitations, and conclusion
Interpret the actual tables and charts produced above. Compare class distributions, feature distributions, and model metrics. Do not claim a model is production-ready based on this small benchmark dataset. A stronger real-world project would validate on newer, independently collected samples and examine measurement and sampling bias.

### Portfolio evidence checklist
- [ ] Run every cell from top to bottom without errors.
- [ ] Review the actual metrics and explain which model performed better and why.
- [ ] Save at least four charts and include them in the report or portfolio.
- [ ] Modify at least one SQL query and explain its output.
- [ ] Be prepared to explain train/test split, standardization, macro-F1, and cross-validation.

**Dataset source:** scikit-learn `load_wine` dataset; original source: UCI Machine Learning Repository, Wine recognition data. See https://archive.ics.uci.edu/dataset/109/wine and https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html.